# Chia train/val/test, cân bằng & Cross-Validation — trên 2 dataset thật

Dùng đúng **2 file CSV** trong zip (`ViHSD.csv`, `Hanoi_housing_dataset.csv`).

| | PHẦN A — Phân loại | PHẦN B — Hồi quy |
|---|---|---|
| Dataset | ViHSD (bình luận MXH) | Giá nhà Hà Nội |
| Target | 3 lớp `CLEAN/OFFENSIVE/HATE` (mất cân bằng) | `Giá/m²` (số liên tục) |
| Cân bằng | Có (oversample train) | Không (hồi quy) |
| **Loại Fold** | **StratifiedKFold** | **KFold** (+ TimeSeriesSplit) |

> **Quy tắc vàng:** mọi bước **học tham số từ dữ liệu** (`fit` vectorizer/scaler/encoder, cân bằng bằng nhân bản) **chỉ được nhìn TRAIN**. `val`/`test` chỉ đi qua `transform`. Lý do: val/test phải đóng vai "dữ liệu chưa từng thấy" → nếu cho chúng tham gia `fit`, điểm đánh giá **đẹp ảo** (data leakage), ra thực tế là sập.

## Hình dung thực tế — 2 sản phẩm + phép ẩn dụ "kỳ thi"

Đừng nghĩ đây là bài tập khô khan. Hai dataset = **hai tính năng sản phẩm thật**:
- **ViHSD** → bộ **lọc bình luận độc hại** cho một app mạng xã hội VN (tự ẩn comment CLEAN/OFFENSIVE/HATE).
- **Hanoi housing** → tính năng **gợi ý giá** khi user đăng tin bán nhà trên app bất động sản.

### Ẩn dụ kỳ thi (nhớ cả đời)
- **train** = *sách bài tập* để ôn.
- **val** = *đề thi thử* — tự chấm để chỉnh cách học, chọn "tủ" (chọn model/tham số).
- **test** = *kỳ thi thật* — **thi đúng 1 lần**, không được xem trước.
- Lén xem đề thi thật để ôn = **gian lận** → điểm cao nhưng **giả** = đúng nghĩa *data leakage*.

### Vì sao "chia trước, cân bằng sau, chỉ trên train"
- Ngoài đời app nhận **~82% bình luận sạch**. Nếu đem *đề thi thử / kỳ thi thật* (val/test) trộn lại cho **50/50** rồi báo cáo F1 đẹp → sếp tưởng ngon, **deploy lên app gặp 82% sạch là sai bét**. ⇒ **val/test phải giống lưu lượng thật**, tuyệt đối không cân bằng.
- Lớp **HATE/OFFENSIVE hiếm**, model "lười" học → ta **cho nó thấy nhiều ví dụ hiếm hơn lúc luyện** (oversample) — nhưng **chỉ lúc luyện (train)**.

### Vì sao `fit` chỉ trên train
Như **hiệu chỉnh cái cân** bằng số liệu cũ (train). Khách mới tới (test) thì **cân bằng đúng cái cân đó**, không chỉnh lại cân theo khách — nếu chỉnh lại là "đo gian".

### Vì sao K-Fold & chọn loại Fold
- **K-Fold** = thay vì 1 đề thi thử (dễ *hên xui*), làm **5 đề khác nhau** rồi lấy **điểm trung bình ± dao động** → biết **thực lực ổn định** hay ăn may.
- **StratifiedKFold** (ViHSD) = chia 5 đề sao cho **đề nào cũng có đủ câu khó** (lớp HATE hiếm), không để 1 đề toàn câu dễ.
- **TimeSeriesSplit** (giá nhà) = học **giá quá khứ để đoán giá tương lai**; cấm dùng giá tương lai đoán quá khứ — vì lúc deploy **làm gì có dữ liệu tương lai**.
- **GroupKFold** = một người đăng 10 tin nhà na ná nhau → đừng để tin của **cùng người** vừa ở train vừa ở test (học tủ theo người).

> Mục A6 sẽ cho thấy *cái giá của gian lận*: làm sai thứ tự → F1 nhảy lên **0.94 ảo**; làm đúng chỉ **0.62**. Như học thuộc đáp án đề thi thử rồi vào thi thật là rớt.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, TimeSeriesSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import f1_score, classification_report, mean_squared_error, r2_score
from sklearn.utils import resample

SEED = 42
LABELS = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}
BASE = "2611328 - Nguyễn Phương Anh Tú - Exploratory Data Analysis (EDA)"

def find_csv(name):
    """Tìm CSV: thư mục hiện tại, rồi thư mục b1."""
    for p in (Path(name), Path("..") / "b1" / BASE / name, Path(BASE) / name):
        if p.exists():
            return p
    raise FileNotFoundError(name)

def show_dist(y, tag=""):
    """In số lượng + tỉ lệ từng lớp (dùng cho bài phân loại)."""
    vc = y.value_counts().sort_index()
    pct = (y.value_counts(normalize=True).sort_index() * 100).round(1)
    body = "   ".join(f"{LABELS[k]}:{int(vc[k])} ({pct[k]}%)" for k in vc.index)
    print(f"  {tag:10s} n={len(y):6d} | {body}")

pd.set_option("display.max_colwidth", 100)

## 0. Nên dùng loại Fold nào? — và VÌ SAO

| Loại Fold | Dùng khi | Vì sao |
|---|---|---|
| **KFold** | Hồi quy / dữ liệu i.i.d., không có lớp | Chia ngẫu nhiên K khối bằng nhau — đơn giản, không thiên lệch khi dữ liệu độc lập. |
| **StratifiedKFold** | **Phân loại**, nhất là **mất cân bằng** | Giữ **tỉ lệ lớp** ở mỗi fold. Nếu dùng KFold thường, fold có thể **thiếu lớp hiếm** → điểm dao động mạnh, ước lượng sai. |
| **GroupKFold** | Có **nhóm/thực thể trùng** (cùng user, cùng căn nhà, cùng phiên) | Giữ cả nhóm về **một phía** → tránh model "thấy" nhóm đó ở cả train lẫn val (leakage theo nhóm). |
| **TimeSeriesSplit** | Dữ liệu có **thời gian**, bài forecasting | Train = **quá khứ**, val = **tương lai**, **không shuffle**. KFold ngẫu nhiên sẽ cho model "thấy tương lai" → leakage thời gian. |

**Áp dụng cho 2 dataset ở đây:**
- **ViHSD** → `StratifiedKFold`. *Vì sao:* lớp `OFFENSIVE` chỉ ~6.8%; KFold thường dễ tạo fold lệch/thiếu lớp này → F1 nhảy loạn. Stratified giữ đúng tỉ lệ 3 lớp ở mọi fold.
- **Hanoi housing** → `KFold` (target liên tục, không có lớp để phân tầng). *Nhưng:* dataset có cột **`Ngày`** → nếu mục tiêu là **dự đoán giá tương lai**, đúng hơn phải dùng `TimeSeriesSplit` (xem cuối Phần B).

# PHẦN A — Phân loại ViHSD (StratifiedKFold)

## A1. Nạp & làm sạch
Bỏ bình luận trùng lặp — nếu để trùng, bản sao có thể rơi vào cả train lẫn test (một dạng leakage).

In [2]:
df = pd.read_csv(find_csv("ViHSD.csv"))
print("Kích thước gốc:", df.shape)
df = df.drop_duplicates(subset="free_text").reset_index(drop=True)
print("Sau khi bỏ trùng:", df.shape)
show_dist(df["label_id"], "TOÀN BỘ")
# File có sẵn cột split (data thực tế đôi khi đã chia sẵn) — ở đây ta tự chia để học:
print("Cột split có sẵn:", df["split"].value_counts().to_dict())
df.head(3)

Kích thước gốc: (33398, 3)
Sau khi bỏ trùng: (30602, 3)
  TOÀN BỘ    n= 30602 | CLEAN:25168 (82.2%)   OFFENSIVE:2093 (6.8%)   HATE:3341 (10.9%)
Cột split có sẵn: {'train': 22557, 'test': 5713, 'dev': 2332}


,free_text,label_id,split
0,Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cute coi mấy giờ này quá hợp lí =]]],0,train
1,Đúng là bọn mắt híp lò xo thụt :))) bên việt nam t cái này ra cách đây 10 năm r và bọn t gọi là ...,2,train
2,Đậu Văn Cường giờ giống thằng sida hơn à,0,train


## A2. Bước 1 — Chia `test` (held-out) + `train`/`val`, có `stratify`
`train_test_split` chỉ tách 2 phần/lần → cắt **2 lần**. `stratify=y` để mọi tập giữ đúng tỉ lệ 3 lớp.
*Vì sao chia trước:* để val/test thật sự "chưa từng thấy"; mọi xử lý sau chỉ học trên train.

In [3]:
X = df["free_text"]
y = df["label_id"]

# Lần 1: tách TEST 20% — KHOÁ lại, chỉ chạm ở cuối
X_trainfull, X_test, y_trainfull, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
# Lần 2: từ 80% còn lại tách VAL (0.25 * 0.80 = 0.20 tổng)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainfull, y_trainfull, test_size=0.25, stratify=y_trainfull, random_state=SEED)

print("Tỉ lệ 3 lớp GIỮ NGUYÊN ở mọi tập nhờ stratify:\n")
for tag, yy in [("TOÀN BỘ", y), ("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    show_dist(yy, tag)

Tỉ lệ 3 lớp GIỮ NGUYÊN ở mọi tập nhờ stratify:

  TOÀN BỘ    n= 30602 | CLEAN:25168 (82.2%)   OFFENSIVE:2093 (6.8%)   HATE:3341 (10.9%)
  TRAIN      n= 18360 | CLEAN:15100 (82.2%)   OFFENSIVE:1255 (6.8%)   HATE:2005 (10.9%)
  VAL        n=  6121 | CLEAN:5034 (82.2%)   OFFENSIVE:419 (6.8%)   HATE:668 (10.9%)
  TEST       n=  6121 | CLEAN:5034 (82.2%)   OFFENSIVE:419 (6.8%)   HATE:668 (10.9%)


## A3. Bước 2 — Cân bằng CHỈ trên `train`
Oversample (nhân bản) lớp thiểu số cho bằng lớp đa số — **chỉ train**.
*Vì sao chỉ train:* `val/test` phải giữ phân phối gốc thì F1 mới trung thực; cân bằng cả val/test = bóp méo thực tế cần đo, và là leakage (sinh mẫu trên dữ liệu đáng lẽ chưa thấy).

In [4]:
def oversample_text(X_text, y, seed=SEED):
    """Nhân bản lớp thiểu số cho bằng lớp đa số (random oversampling)."""
    d = pd.DataFrame({"text": np.asarray(X_text), "y": np.asarray(y)})
    n_max = d["y"].value_counts().max()
    parts = [(resample(g, replace=True, n_samples=n_max, random_state=seed) if len(g) < n_max else g)
             for _, g in d.groupby("y")]
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out["text"], out["y"]

print("TRƯỚC cân bằng:"); show_dist(y_train, "TRAIN")
X_train_bal, y_train_bal = oversample_text(X_train, y_train)
print("\nSAU cân bằng (chỉ train — 3 lớp bằng nhau):"); show_dist(y_train_bal, "TRAIN_BAL")
print("\nVAL & TEST giữ NGUYÊN phân phối gốc:"); show_dist(y_val, "VAL"); show_dist(y_test, "TEST")

TRƯỚC cân bằng:
  TRAIN      n= 18360 | CLEAN:15100 (82.2%)   OFFENSIVE:1255 (6.8%)   HATE:2005 (10.9%)

SAU cân bằng (chỉ train — 3 lớp bằng nhau):
  TRAIN_BAL  n= 45300 | CLEAN:15100 (33.3%)   OFFENSIVE:15100 (33.3%)   HATE:15100 (33.3%)

VAL & TEST giữ NGUYÊN phân phối gốc:
  VAL        n=  6121 | CLEAN:5034 (82.2%)   OFFENSIVE:419 (6.8%)   HATE:668 (10.9%)
  TEST       n=  6121 | CLEAN:5034 (82.2%)   OFFENSIVE:419 (6.8%)   HATE:668 (10.9%)


## A4. Bước 3–5 — Vector hoá (fit train) → huấn luyện → đánh giá
`TfidfVectorizer` học từ vựng (`fit`) **chỉ trên train đã cân bằng**, rồi `transform` val/test. Đây chính là `fit_transform(train)` vs `transform(val/test)`.

In [5]:
vec = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(X_train_bal)   # fit_transform: HỌC từ vựng trên TRAIN
Xva = vec.transform(X_val)             # transform: dùng lại từ vựng TRAIN
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(Xtr, y_train_bal)
f1_val  = f1_score(y_val,  clf.predict(Xva), average="macro")
f1_test = f1_score(y_test, clf.predict(Xte), average="macro")
print(f"F1-macro VAL : {f1_val:.4f}  (chọn model/hyperparameter)")
print(f"F1-macro TEST: {f1_test:.4f}  (báo cáo cuối, chạm 1 lần)\n")
print(classification_report(y_test, clf.predict(Xte),
                            target_names=[LABELS[i] for i in (0, 1, 2)]))

F1-macro VAL : 0.6349  (chọn model/hyperparameter)
F1-macro TEST: 0.6230  (báo cáo cuối, chạm 1 lần)

              precision    recall  f1-score   support

       CLEAN       0.94      0.89      0.91      5034
   OFFENSIVE       0.37      0.39      0.38       419
        HATE       0.50      0.67      0.57       668

    accuracy                           0.83      6121
   macro avg       0.60      0.65      0.62      6121
weighted avg       0.85      0.83      0.84      6121



## A5. `StratifiedKFold` — IN RA TỪNG FOLD
Thay 1 tập val cố định bằng **K khối**. Mỗi vòng: 1 fold làm val, K−1 fold còn lại làm train; lặp K lần → **mỗi mẫu được validate đúng 1 lần**.
*Vì sao Stratified ở đây:* lớp `OFFENSIVE` hiếm → cần giữ tỉ lệ ở mỗi fold, nếu không có fold thiếu lớp này.
**Quan trọng:** cân bằng + fit vectorizer đặt **trong** vòng lặp, **chỉ** trên train-fold; `test` vẫn held-out.

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
Xtf = X_trainfull.reset_index(drop=True)   # CV chạy trên train-full; TEST khoá ngoài
ytf = y_trainfull.reset_index(drop=True)

clf_scores = []
for fold, (tr, va) in enumerate(skf.split(Xtf, ytf), 1):
    Xtr_f, ytr_f = Xtf.iloc[tr], ytf.iloc[tr]
    Xva_f, yva_f = Xtf.iloc[va], ytf.iloc[va]
    print(f"========== FOLD {fold}/5 ==========")
    print(f"  train-fold={len(tr):6d} | val-fold={len(va):6d}")
    show_dist(ytr_f, "train"); show_dist(yva_f, "val")

    # THỨ TỰ ĐÚNG TRONG FOLD: cân bằng + fit vectorizer chỉ train-fold
    Xtr_fb, ytr_fb = oversample_text(Xtr_f, ytr_f)
    v = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2)
    m = LogisticRegression(max_iter=1000)
    m.fit(v.fit_transform(Xtr_fb), ytr_fb)
    f1 = f1_score(yva_f, m.predict(v.transform(Xva_f)), average="macro")
    clf_scores.append(f1)
    print(f"  -> F1-macro fold {fold}: {f1:.4f}\n")

print("Điểm từng fold:", [round(s, 4) for s in clf_scores])
print(f"CV F1-macro: {np.mean(clf_scores):.4f} ± {np.std(clf_scores):.4f}")

========== FOLD 1/5 ==========
  train-fold= 19584 | val-fold=  4897
  train      n= 19584 | CLEAN:16107 (82.2%)   OFFENSIVE:1339 (6.8%)   HATE:2138 (10.9%)
  val        n=  4897 | CLEAN:4027 (82.2%)   OFFENSIVE:335 (6.8%)   HATE:535 (10.9%)


  -> F1-macro fold 1: 0.6210

========== FOLD 2/5 ==========
  train-fold= 19585 | val-fold=  4896
  train      n= 19585 | CLEAN:16107 (82.2%)   OFFENSIVE:1340 (6.8%)   HATE:2138 (10.9%)
  val        n=  4896 | CLEAN:4027 (82.3%)   OFFENSIVE:334 (6.8%)   HATE:535 (10.9%)


  -> F1-macro fold 2: 0.6373

========== FOLD 3/5 ==========
  train-fold= 19585 | val-fold=  4896
  train      n= 19585 | CLEAN:16107 (82.2%)   OFFENSIVE:1339 (6.8%)   HATE:2139 (10.9%)
  val        n=  4896 | CLEAN:4027 (82.3%)   OFFENSIVE:335 (6.8%)   HATE:534 (10.9%)


  -> F1-macro fold 3: 0.6131

========== FOLD 4/5 ==========
  train-fold= 19585 | val-fold=  4896
  train      n= 19585 | CLEAN:16107 (82.2%)   OFFENSIVE:1339 (6.8%)   HATE:2139 (10.9%)
  val        n=  4896 | CLEAN:4027 (82.3%)   OFFENSIVE:335 (6.8%)   HATE:534 (10.9%)


  -> F1-macro fold 4: 0.6264

========== FOLD 5/5 ==========
  train-fold= 19585 | val-fold=  4896
  train      n= 19585 | CLEAN:16108 (82.2%)   OFFENSIVE:1339 (6.8%)   HATE:2138 (10.9%)
  val        n=  4896 | CLEAN:4026 (82.2%)   OFFENSIVE:335 (6.8%)   HATE:535 (10.9%)


  -> F1-macro fold 5: 0.6180

Điểm từng fold: [0.621, 0.6373, 0.6131, 0.6264, 0.618]
CV F1-macro: 0.6231 ± 0.0083


### Giải thích từng fold
- Mỗi vòng đổi khối val khác nhau → sau 5 vòng **mọi mẫu đều từng ở val đúng 1 lần**.
- **Tỉ lệ lớp in ra ở mỗi fold gần như nhau** → tác dụng của `Stratified` (lý do chọn nó cho dữ liệu lệch lớp).
- **`mean ± std`:** trung bình = ước lượng ổn định hơn 1 lần chia; **std** cho biết model nhạy với cách chia tới đâu (std lớn = variance cao, chưa ổn định).
- Cân bằng + fit vectorizer **trong** fold → val-fold luôn sạch, không leakage.

## A6. Làm SAI thứ tự → leakage (đo cụ thể)
Cân bằng (nhân bản) **TRƯỚC khi chia fold**: bản sao giống hệt rơi vào cả train-fold lẫn val-fold → model học thuộc → F1 thổi phồng.

In [7]:
X_all_bal, y_all_bal = oversample_text(Xtf, ytf)          # SAI: nhân bản toàn bộ TRƯỚC
X_all_bal = X_all_bal.reset_index(drop=True); y_all_bal = y_all_bal.reset_index(drop=True)
wrong = []
for tr, va in skf.split(X_all_bal, y_all_bal):
    v = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=2)
    m = LogisticRegression(max_iter=1000)
    m.fit(v.fit_transform(X_all_bal.iloc[tr]), y_all_bal.iloc[tr])
    wrong.append(f1_score(y_all_bal.iloc[va], m.predict(v.transform(X_all_bal.iloc[va])), average="macro"))
print(f"ĐÚNG (cân bằng TRONG fold)      : {np.mean(clf_scores):.4f}")
print(f"SAI  (cân bằng TRƯỚC khi chia)  : {np.mean(wrong):.4f}  <- thổi phồng do bản sao rò rỉ sang val")
print(f"Chênh lệch ảo: +{np.mean(wrong) - np.mean(clf_scores):.4f} F1-macro")

ĐÚNG (cân bằng TRONG fold)      : 0.6231
SAI  (cân bằng TRƯỚC khi chia)  : 0.9378  <- thổi phồng do bản sao rò rỉ sang val
Chênh lệch ảo: +0.3147 F1-macro


# PHẦN B — Hồi quy giá nhà Hà Nội (KFold)

Khác Phần A: target `Giá/m²` **liên tục** → **không cân bằng lớp**, và dùng **`KFold`** (không Stratified, vì không có lớp). Tiền xử lý đổi từ TF-IDF sang **`StandardScaler`** (số) + **`OneHotEncoder`** (Quận, loại hình) — vẫn `fit` **chỉ trên train**.

## B1. Nạp & làm sạch (chuyển text "triệu/m²", "tỷ/m²"... về số)

In [8]:
dfh = pd.read_csv(find_csv("Hanoi_housing_dataset.csv")).drop(columns=["Unnamed: 0"])
print("Kích thước gốc:", dfh.shape)

def to_number(s):
    if pd.isna(s): return np.nan
    m = re.search(r"[0-9]+(?:[.][0-9]+)?", str(s).replace(".", "").replace(",", "."))
    return float(m.group(0)) if m else np.nan

def to_price_m2(s):           # quy mọi đơn vị về TRIỆU đồng/m²
    if pd.isna(s): return np.nan
    v, t = to_number(s), str(s)
    if "tỷ" in t: return v * 1000
    if "đ/m" in t and "triệu" not in t: return v / 1e6
    return v

dfh["area"] = dfh["Diện tích"].apply(to_number)
dfh["bedrooms"] = dfh["Số phòng ngủ"].apply(to_number)
dfh["price_m2"] = dfh["Giá/m2"].apply(to_price_m2)
dfh["district"] = dfh["Quận"].fillna("NA")
dfh["house_type"] = dfh["Loại hình nhà ở"].fillna("NA")
dfh["date"] = pd.to_datetime(dfh["Ngày"], errors="coerce")

# Lọc lỗi đơn vị / ngoại lai cực đoan về khoảng hợp lý của Hà Nội
n0 = len(dfh)
dfh = dfh[dfh["price_m2"].between(5, 500) & dfh["area"].between(10, 1000)].copy()
dfh = dfh.dropna(subset=["area", "bedrooms", "price_m2"]).reset_index(drop=True)
print(f"Sau làm sạch: {len(dfh)} dòng (loại {n0 - len(dfh)} dòng lỗi/thiếu)")
dfh[["area", "bedrooms", "price_m2"]].describe().round(1)

Kích thước gốc: (82497, 12)


Sau làm sạch: 81076 dòng (loại 1421 dòng lỗi/thiếu)


,area,bedrooms,price_m2
count,81076.0,81076.0,81076.0
mean,47.2,3.9,100.4
std,32.9,1.4,51.1
min,10.0,1.0,5.0
25%,34.0,3.0,73.3
50%,40.0,4.0,90.0
75%,50.0,4.0,110.3
max,951.0,10.0,500.0


## B2. Chia train/test + định nghĩa pipeline tiền xử lý
`KFold` **không** stratify (vì target liên tục). Pipeline = `ColumnTransformer` (`StandardScaler` cho số + `OneHotEncoder` cho Quận/loại hình) → `Ridge`. Đặt trong **Pipeline** để mỗi fold `fit` preprocessing **chỉ trên train-fold** (chống leakage tự động).

In [9]:
NUM = ["area", "bedrooms"]
CAT = ["district", "house_type"]
Xh = dfh[NUM + CAT]
yh = dfh["price_m2"]

# Test held-out 20% (KFold không stratify -> chia ngẫu nhiên)
Xh_tf, Xh_te, yh_tf, yh_te = train_test_split(Xh, yh, test_size=0.20, random_state=SEED)
Xh_tf = Xh_tf.reset_index(drop=True); yh_tf = yh_tf.reset_index(drop=True)

def make_pipe():
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM),                                   # fit chỉ train-fold
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),             # fit chỉ train-fold
    ])
    return Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])

print("train-full:", len(Xh_tf), "| test:", len(Xh_te))

train-full: 64860 | test: 16216


## B3. `KFold` — IN RA TỪNG FOLD (RMSE + R²)
*Vì sao KFold (không Stratified):* không có "lớp" để giữ tỉ lệ; mỗi tin nhà coi như độc lập. Báo cáo **RMSE** (sai số, triệu/m²) và **R²** (mức giải thích phương sai) cho từng fold.

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
reg_rmse, reg_r2 = [], []
for fold, (tr, va) in enumerate(kf.split(Xh_tf), 1):
    pipe = make_pipe().fit(Xh_tf.iloc[tr], yh_tf.iloc[tr])     # fit (gồm scaler/onehot) CHỈ train-fold
    pred = pipe.predict(Xh_tf.iloc[va])
    rmse = mean_squared_error(yh_tf.iloc[va], pred) ** 0.5
    r2 = r2_score(yh_tf.iloc[va], pred)
    reg_rmse.append(rmse); reg_r2.append(r2)
    print(f"FOLD {fold}/5 | train={len(tr):6d} val={len(va):6d} | RMSE={rmse:6.2f} triệu/m²  R²={r2:.3f}")
print(f"\nKFold RMSE: {np.mean(reg_rmse):.2f} ± {np.std(reg_rmse):.2f} triệu/m²")
print(f"KFold R²  : {np.mean(reg_r2):.3f} ± {np.std(reg_r2):.3f}")

FOLD 1/5 | train= 51888 val= 12972 | RMSE= 42.54 triệu/m²  R²=0.323


FOLD 2/5 | train= 51888 val= 12972 | RMSE= 42.69 triệu/m²  R²=0.301


FOLD 3/5 | train= 51888 val= 12972 | RMSE= 41.82 triệu/m²  R²=0.325


FOLD 4/5 | train= 51888 val= 12972 | RMSE= 43.18 triệu/m²  R²=0.309


FOLD 5/5 | train= 51888 val= 12972 | RMSE= 41.30 triệu/m²  R²=0.308

KFold RMSE: 42.30 ± 0.67 triệu/m²
KFold R²  : 0.313 ± 0.009


### Giải thích
- R² ở mức vừa phải vì mới dùng vài đặc trưng (diện tích, phòng ngủ, quận, loại hình); **giá/m² phụ thuộc nhiều vào vị trí chi tiết** — thêm đặc trưng sẽ cải thiện. Trọng tâm ở đây là **quy trình CV**, không phải tối ưu điểm.
- RMSE/R² **ổn định giữa các fold** (std nhỏ) → ước lượng đáng tin, model không quá nhạy với cách chia.

## B4. `TimeSeriesSplit` — vì dataset có cột `Ngày`
*Vì sao cần xét:* nếu bài toán là **dự đoán giá tương lai**, dùng KFold ngẫu nhiên sẽ cho model học từ tin **tương lai** để đoán **quá khứ** → leakage thời gian, điểm ảo. `TimeSeriesSplit` ép **train = quá khứ, val = tương lai**, không shuffle.

In ra từng fold sẽ thấy: val-fold luôn nằm **sau** train-fold về thời gian, và train lớn dần.

In [11]:
# Chỉ lấy phần có ngày hợp lệ, SẮP XẾP theo thời gian (bắt buộc cho TimeSeriesSplit)
dft = dfh.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
print(f"Số tin có ngày hợp lệ: {len(dft)}  ({dft['date'].min().date()} -> {dft['date'].max().date()})")

Xt = dft[NUM + CAT]; yt = dft["price_m2"]
tss = TimeSeriesSplit(n_splits=5)
ts_rmse = []
for fold, (tr, va) in enumerate(tss.split(Xt), 1):
    pipe = make_pipe().fit(Xt.iloc[tr], yt.iloc[tr])
    rmse = mean_squared_error(yt.iloc[va], pipe.predict(Xt.iloc[va])) ** 0.5
    ts_rmse.append(rmse)
    print(f"FOLD {fold}/5 | train={len(tr):6d} (đến {dft['date'].iloc[tr[-1]].date()}) "
          f"| val={len(va):5d} ({dft['date'].iloc[va[0]].date()} -> {dft['date'].iloc[va[-1]].date()}) "
          f"| RMSE={rmse:6.2f}")
print(f"\nTimeSeriesSplit RMSE: {np.mean(ts_rmse):.2f} ± {np.std(ts_rmse):.2f} triệu/m²")

Số tin có ngày hợp lệ: 81076  (2019-08-05 -> 2020-08-05)


FOLD 1/5 | train= 13516 (đến 2020-06-17) | val=13512 (2020-06-17 -> 2020-06-28) | RMSE= 42.72
FOLD 2/5 | train= 27028 (đến 2020-06-28) | val=13512 (2020-06-28 -> 2020-07-09) | RMSE= 43.35


FOLD 3/5 | train= 40540 (đến 2020-07-09) | val=13512 (2020-07-09 -> 2020-07-20) | RMSE= 43.05


FOLD 4/5 | train= 54052 (đến 2020-07-20) | val=13512 (2020-07-20 -> 2020-07-28) | RMSE= 41.43


FOLD 5/5 | train= 67564 (đến 2020-07-28) | val=13512 (2020-07-28 -> 2020-08-05) | RMSE= 39.53

TimeSeriesSplit RMSE: 42.02 ± 1.41 triệu/m²


### So sánh KFold vs TimeSeriesSplit
- KFold: ước lượng "trộn thời gian" — hợp lý nếu coi mỗi tin **độc lập** (bài định giá tại 1 thời điểm).
- TimeSeriesSplit: ước lượng **trung thực hơn cho forecasting** vì luôn đoán tương lai từ quá khứ. Train tăng dần, mỗi fold val là một quãng thời gian kế tiếp.
- **Chọn cái nào?** Theo *mục tiêu*: định giá hiện tại → KFold; dự báo giá tương lai → **TimeSeriesSplit**.

## Kết luận chung

1. **CHIA TRƯỚC** mọi thứ: `train/val/test`; `test` khoá lại, chạm 1 lần.
2. **Cân bằng** chỉ trên **train**, **sau** khi chia (Phần A). Hồi quy không cân bằng lớp (Phần B).
3. **`fit`** vectorizer/scaler/encoder **chỉ trên train** (hoặc train-fold), rồi `transform` phần còn lại. Dùng **Pipeline** để tự động đúng thứ tự trong CV.
4. **Chọn loại Fold theo bài toán — và biết lý do:**
   - Phân loại mất cân bằng → **StratifiedKFold** (giữ tỉ lệ lớp).
   - Hồi quy / i.i.d. → **KFold**.
   - Có nhóm trùng → **GroupKFold**. Có thời gian / forecasting → **TimeSeriesSplit**.
5. Sai thứ tự (xử lý/cân bằng trước khi chia) → **data leakage** → điểm đẹp ảo (đo ở A6).

> Một câu: **"Tách test ra trước đã; cái gì cần học từ dữ liệu thì chỉ học trên train; chọn Fold theo đúng bản chất dữ liệu."**